# RAG Demo: Pinecone + Alibaba Cloud Model Studio (Qwen)

This notebook keeps the original **Victoria on Move** RAG scenario, but replaces:

- **Chroma DB → Pinecone**
- **OpenAI embeddings → Alibaba Cloud Model Studio `text-embedding-v4`**
- **OpenAI LLM → Alibaba Cloud Model Studio Qwen**

Alibaba Cloud Model Studio exposes Qwen through an OpenAI-compatible API, so LangChain's OpenAI-compatible client can be used as the transport client. The actual model provider is **Alibaba Cloud Model Studio**, not OpenAI.

> **Important:** Do not put API keys directly in the notebook. Use environment variables or `.env`.


## 1. Install / update the required packages

Run this cell once in a fresh environment. Restart the kernel if the package manager asks you to.


## 2. Imports


In [15]:
import os
from getpass import getpass

from dotenv import load_dotenv

from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore


## 3. Configure Alibaba Cloud Model Studio and Pinecone

### Required credentials

Set these environment variables:

- `DASHSCOPE_API_KEY` = Alibaba Cloud Model Studio API key
- `PINECONE_API_KEY` = Pinecone API key

The default Model Studio endpoint below uses the Singapore region. If your Model Studio workspace is in another region, change `DASHSCOPE_BASE_URL` accordingly.

The embedding model is `text-embedding-v4` with **1024 dimensions**, which is a good general-purpose choice for RAG.


In [16]:
load_dotenv()

# Ask for keys only if they are not already available in the environment.
if not os.getenv("DASHSCOPE_API_KEY"):
    os.environ["DASHSCOPE_API_KEY"] = getpass("Enter Alibaba Cloud Model Studio API key: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter Pinecone API key: ")

# Alibaba Cloud Model Studio OpenAI-compatible endpoint.
# Default: Singapore shared endpoint.
DASHSCOPE_BASE_URL = os.getenv(
    "DASHSCOPE_BASE_URL",
    "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
)

# Qwen text-generation model.
# Change this if a different Qwen model is enabled in your Model Studio account.
QWEN_MODEL = os.getenv("QWEN_MODEL", "qwen-plus")

# Alibaba Cloud Qwen embedding model.
EMBEDDING_MODEL = "text-embedding-v4"
EMBEDDING_DIMENSION = 384

# Pinecone settings
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "victoria-on-move-qwen-rag")
PINECONE_NAMESPACE = os.getenv("PINECONE_NAMESPACE", "victoria-on-move")

print("Qwen model:", QWEN_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding dimension:", EMBEDDING_DIMENSION)
print("Pinecone index:", PINECONE_INDEX_NAME)


Qwen model: qwen-plus
Embedding model: text-embedding-v4
Embedding dimension: 384
Pinecone index: victoria-on-move-qwen-rag


## 4. Load the original live website data

This is the same data source used in the original notebook.


In [17]:
urls = [
    "https://www.victoriaonmove.com.au/local-removalists.html",
    "https://victoriaonmove.com.au/index.html",
    "https://victoriaonmove.com.au/contact.html",
]

loader = UnstructuredURLLoader(urls=urls)
data = loader.load()

print("Loaded documents:", len(data))


Loaded documents: 3


In [18]:
data

[Document(metadata={'source': 'https://www.victoriaonmove.com.au/local-removalists.html'}, page_content="Local Removalists Melbourne\n\nTop-notch local moving services tailored to your needs — from studio apartments to large family homes.\n\nWhat We Do\n\nComprehensive Local Moving Services\n\nWe offer a full range of local moving services for Melbourne residents and businesses. Our experienced team handles every move with care and professionalism.\n\nApartment Moving\n\nEfficient and careful relocation services tailored for apartments of all sizes. We navigate lifts, stairs, and narrow corridors with ease.\n\nVilla Moving\n\nComprehensive moving solutions for large residences and villas. We handle your valuable possessions with the utmost care and attention.\n\nHousehold Moving\n\nFull-service moving options for households of every size, including packing, loading, transportation, and unpacking at your new home.\n\nOffice Moving\n\nSpecialised expertise in office relocations. We minim

## 5. Split the documents into chunks


In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = text_splitter.split_documents(data)

print("Total number of chunks:", len(docs))


Total number of chunks: 14


In [20]:
docs[0]

Document(metadata={'source': 'https://www.victoriaonmove.com.au/local-removalists.html'}, page_content='Local Removalists Melbourne\n\nTop-notch local moving services tailored to your needs — from studio apartments to large family homes.\n\nWhat We Do\n\nComprehensive Local Moving Services\n\nWe offer a full range of local moving services for Melbourne residents and businesses. Our experienced team handles every move with care and professionalism.\n\nApartment Moving\n\nEfficient and careful relocation services tailored for apartments of all sizes. We navigate lifts, stairs, and narrow corridors with ease.\n\nVilla Moving\n\nComprehensive moving solutions for large residences and villas. We handle your valuable possessions with the utmost care and attention.\n\nHousehold Moving\n\nFull-service moving options for households of every size, including packing, loading, transportation, and unpacking at your new home.\n\nOffice Moving\n\nSpecialised expertise in office relocations. We minimi

## 6. Create Alibaba Cloud Qwen embeddings

`text-embedding-v4` is Alibaba Cloud Model Studio's Qwen3-Embedding-series text embedding model. We use 1024 dimensions, so the Pinecone index must also be created with dimension `1024`.


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
import os

# Load Hugging Face embedding model
embeddings = HuggingFaceEmbeddings(
    model_name=os.environ["HF_EMBEDDING_MODEL"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Quick embedding test
test_embedding = embeddings.embed_query(
    "What services does Victoria on Move provide?"
)

print("Embedding vector length:", len(test_embedding))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5095.40it/s]


Embedding vector length: 384


## 7. Create / connect to the Pinecone index

The index dimension is matched to the Alibaba Cloud `text-embedding-v4` output dimension.

If an index with the same name already exists but has a different dimension, use a new index name or recreate the old index in Pinecone.


In [22]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )
    print("Created Pinecone index:", PINECONE_INDEX_NAME)
else:
    print("Using existing Pinecone index:", PINECONE_INDEX_NAME)

index = pc.Index(PINECONE_INDEX_NAME)


Using existing Pinecone index: victoria-on-move-qwen-rag


## 8. Store the document chunks in Pinecone


In [23]:
from langchain_pinecone import PineconeVectorStore

# Create Pinecone vector store
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,
)

# Upload document chunks
vectorstore.add_documents(
    documents=docs,
    namespace=PINECONE_NAMESPACE,
)

print("Documents successfully stored in Pinecone.")
print("Namespace:", PINECONE_NAMESPACE)
print("Number of chunks:", len(docs))

Documents successfully stored in Pinecone.
Namespace: victoria-on-move
Number of chunks: 14


## 9. Create the retriever


In [24]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retrieved_docs = retriever.invoke(
    "What kind of services do Victoria on Move provide?"
)

print("Retrieved documents:", len(retrieved_docs))


Retrieved documents: 3


In [25]:
retrieved_docs

[Document(id='33e2eab4-b6e8-45d8-b8eb-4d5bae30ddef', metadata={'source': 'https://www.victoriaonmove.com.au/local-removalists.html'}, page_content='Local Removalists Melbourne\n\nTop-notch local moving services tailored to your needs — from studio apartments to large family homes.\n\nWhat We Do\n\nComprehensive Local Moving Services\n\nWe offer a full range of local moving services for Melbourne residents and businesses. Our experienced team handles every move with care and professionalism.\n\nApartment Moving\n\nEfficient and careful relocation services tailored for apartments of all sizes. We navigate lifts, stairs, and narrow corridors with ease.\n\nVilla Moving\n\nComprehensive moving solutions for large residences and villas. We handle your valuable possessions with the utmost care and attention.\n\nHousehold Moving\n\nFull-service moving options for households of every size, including packing, loading, transportation, and unpacking at your new home.\n\nOffice Moving\n\nSpecialise

In [26]:
print(retrieved_docs[0].page_content)

Local Removalists Melbourne

Top-notch local moving services tailored to your needs — from studio apartments to large family homes.

What We Do

Comprehensive Local Moving Services

We offer a full range of local moving services for Melbourne residents and businesses. Our experienced team handles every move with care and professionalism.

Apartment Moving

Efficient and careful relocation services tailored for apartments of all sizes. We navigate lifts, stairs, and narrow corridors with ease.

Villa Moving

Comprehensive moving solutions for large residences and villas. We handle your valuable possessions with the utmost care and attention.

Household Moving

Full-service moving options for households of every size, including packing, loading, transportation, and unpacking at your new home.

Office Moving

Specialised expertise in office relocations. We minimise downtime and ensure your business is up and running as quickly as possible.

Furniture Moving


## 10. Connect Qwen from Alibaba Cloud Model Studio

Model Studio provides an OpenAI-compatible API. `ChatOpenAI` is used here only as the LangChain-compatible client interface; the request is sent to the Alibaba Cloud Model Studio endpoint and the selected model is Qwen.


In [27]:
llm = ChatOpenAI(
    model=QWEN_MODEL,
    api_key=os.environ["DASHSCOPE_API_KEY"],
    base_url=DASHSCOPE_BASE_URL,
    temperature=0.4,
    max_tokens=500,
)

# Optional direct Qwen test
qwen_test = llm.invoke("In one sentence, what is RAG?")
print(qwen_test.content)


RAG (Retrieval-Augmented Generation) is an AI technique that enhances large language models by dynamically retrieving relevant information from external knowledge sources (e.g., databases or documents) and incorporating it into the prompt to generate more accurate, up-to-date, and contextually grounded responses.


## 11. Build the RAG chain


In [28]:

from langchain_core.prompts import ChatPromptTemplate

# --------------------------------------------------
# RAG Prompt
# --------------------------------------------------

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following retrieved context to answer the question. "
    "If the answer is not present in the context, say that you do not know. "
    "Do not invent information. "
    "Keep the answer concise, with a maximum of three sentences."
    "\n\n"
    "Retrieved context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


# --------------------------------------------------
# Helper function to combine retrieved documents
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# --------------------------------------------------
# RAG Function
# --------------------------------------------------

def rag_chain(question):

    # 1. Retrieve relevant documents from Pinecone
    retrieved_docs = retriever.invoke(question)

    # 2. Convert documents into context
    context = format_docs(retrieved_docs)

    # 3. Send question + context to Qwen
    response = llm.invoke(
        prompt.format_messages(
            context=context,
            input=question
        )
    )

    # 4. Return answer
    return response.content


# --------------------------------------------------
# Test the RAG system
# --------------------------------------------------

question = "What kind of services do they provide?"

answer = rag_chain(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)


Question:
What kind of services do they provide?

Answer:
They provide comprehensive local moving services for Melbourne residents and businesses, including apartment moving, villa moving, household moving, office moving, and furniture moving. Services cover packing, loading, transportation, and unpacking, with care tailored to different property sizes—from studio apartments to large family homes and offices. Their team is experienced and professional, ensuring minimal downtime and safe handling of valuable possessions.


## 12. Ask a question

The complete RAG flow is now:

**Website → Documents → Chunks → Qwen Embeddings → Pinecone → Retriever → Retrieved Context → Qwen → Answer**


In [23]:

question = "What kind of services do Victoria on Move provide?"

answer = rag_chain(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)


Question:
What kind of services do Victoria on Move provide?

Answer:
Victoria on Move provides comprehensive local moving services in Melbourne, including apartment, villa, household, office, and furniture moving. They offer full-service options such as packing, loading, transportation, and unpacking. Their services are tailored for both residents and businesses, with transparent pricing and a focus on care and professionalism.


## 13. Try more questions

Change the questions below to test the retrieval and Qwen answer generation.


In [24]:

questions = [
    "What types of trucks do they offer?",
    "Do they provide packing and unpacking services?",
    "What cities can they move to from Melbourne?",
    "What insurance do they provide?",
]

for question in questions:

    # Get answer from the RAG chain
    answer = rag_chain(question)

    print("\nQuestion:", question)
    print("Answer:", answer)




Question: What types of trucks do they offer?
Answer: They offer five types of trucks: Small (4.5 ton), Medium (6 ton), Large (8 ton), X-Large (10 ton), and the Biggest Truck (12 ton). Each is suited for different move sizes—from student apartments to 5+ bedroom homes and large office relocations. All trucks come with 2 professional movers, trolleys, moving blankets, and loading ramps.

Question: Do they provide packing and unpacking services?
Answer: Yes, they provide optional packing and unpacking services using quality materials to ensure items arrive safely. These services are mentioned multiple times across the context as part of their home removals offerings.

Question: What cities can they move to from Melbourne?
Answer: Victoria On Move can move you from Melbourne to any major Australian city, including Sydney and Brisbane, as explicitly mentioned in the context.

Question: What insurance do they provide?
Answer: They provide transit and public liability insurance on every mov

## RAG architecture used in this notebook

1. **Data source:** Victoria on Move website
2. **Loader:** `UnstructuredURLLoader`
3. **Chunking:** `RecursiveCharacterTextSplitter`
4. **Embedding model:** Alibaba Cloud Model Studio `text-embedding-v4`
5. **Vector database:** Pinecone
6. **Retriever:** Pinecone similarity search
7. **LLM:** Alibaba Cloud Model Studio Qwen
8. **Generation:** Qwen answers using retrieved context

This keeps the original notebook scenario while replacing the OpenAI/Chroma components with Alibaba Cloud Model Studio Qwen and Pinecone.
